   In Pintos, condition variables in a monitor are synchronization tools that
   allows threads inside a monitor to wait until a specific logical condition
   becomes true. 

   - DEFITION: A condition variable (`struct condition` in `threads/synch.h`)
     reoresebts a queue of waiting threads tied to a particular state or event
     (such as "buffer is not full" or "data is ready")
   - RELATIONSHIP WITH LOCKS: Condition variables do not have internal counters
     or "history" like semaphores, they require an associated monitor lock to
     control access to shared state. 



---
   looking at a clever operating system trick used to put a thread to sleep
   with surgical precision. instead of putting the thread's main execution
   context directly nto the condition variable's waitlist, the thread creates
   a temporary, personal semaphore called `struct semaphoe_elem waiter` right
   on its own local C stack. It initilaises thsi personal semaphore to 0, 
   pushes its internal list element (`&waiter.elem`) onto the ocndition 
   variable's `waiters` queeu, and then drops the monitor lock and calls
   `sema_down(&waiter.semaphore)`. Because this semaphore's value is 0, this
   immediately forces the thread to put itself to sleep on its own current
   semaphore. 

   This design 

-- ... "dropping a lock" simply means releasing the lock. 

   When the thread "drops" a lock, it relinquishes ownership of that 
   synchronization object, allowing other threads that are waiting on the lock
   to acquire it and enter the critical section. 

   In Pintos, this function is performed by calling `lock_release(struct lock *lock);`

-- 

-- WHY LOCKS ARE ASSOSCIATED WITH CONDITION VARIABLES
   - ATOMIC OPERATIONS: The primary function of a condition variable (`wait`)
     is to automatically release the lock and put the thread to sleep. Without
     this atomicity, a race condition could cause a notification from another
     thread to be missed peermanently. 
   - PROTECTING SHARED STATE: The condition being checked (like a boolean flag
     or queue size) is shared data accessed by multiple threads. The lock 
     protects this shared dstate from data races during reads and writes. 

---

   ... terrible historical naming convention in computer sceicne. A 
   `struct condition` (often called a condition variable) does not actually
   store a mathematical condition or boolean logic. It should have been named a
   `wait_queue` or `sleeping_lounge`. 

```c

int portions cooked = 0;     // The shared state, the logical condition you actually care about
struct lock kitchen_lock;    // The monitor lock, protects the shared state from race condition
struct condition food_ready; // The condition variable, the waiting room for hungry threads

void customer_thread(void) {
    lock_acquire(&kitchen_lock);
    while (portions_cooked == 0) {
        // If false, it parks itself in the `food_ready` waiting room
        cond_wait(&food_ready, &kitchen_lock);
    }
    portions_cooked--; // eats ood
    lock_release(&kitchen_lock); // release resources
}

void chef_thread(void) {
    lock_acquire(&kitchen_lock);
    
    portions_cooked++; // cooks food
    cond_signal(&food_ready, &kitchen_lock);
    
    lock_release(&kitchen_lock);
}


```


   1. The customer drops the lock and falls asleep on `sema_down`
   2. The chef acquires the lock, cooks, and calls `cond_signal`
   3. `cond_signal` calls `sema_up`, which wakes the Customer up and puts it on
      the CPUs ready list.
   4. The Customer resumes inside `cond_wait` and immediately calls `lock_acquire`.
   5. Because the Chef hasn't executed `lock_release` yet, the Customer 
      instantly goes back to sleep--this time waiting for the `kitchen_lock`
      rather than the condition vairable.
   6. A ms later, the Chef executes `lock_release`
   7. This wakes the Customer up again. The Customer successfully acquires the 
      lock, returns from `cond_wait`, and reevaluates the `while` loop. 

---

   Using the `strcpy()` function is a bad idea primaily because it lacks any
   bounds checking for the destination buffer. When you call this function,
   it copes chars from src string until it encounters '\0', completely ignoring
   whether the destination has enough memory allocated to hold all that data.
   If src string ... resulting in buffer overflow. ... memory corruption can 
   cause unpredictable kernel panics, silently overwrite critical system data,
   or create severe security vulnerabilities that allow attackers to hijack a
   program's execution flow.

   Because of these risks, modern coding standards discourage its use, and the 
   PintOS C library intentiooanlly excludes notoriously unsafe function like it. 
   Instead of `strcpy()`, you should use safer alternatives such as `strlcpy()`
   . These modern replacements require you to explicitl pass the maximum size of
   the destination buffer as an argument, ensuring that the copied string is 
   safetly truncated and properly null-terminated if it exceeds the available 
   space, thereby eliminating risk of overflow... 

   WHY THE BUILD LIVES in `src/devices/`. Each task directory builds a DIFFERENT
   KERNEL. Recall conditional compilation from the c recap lecture: the 
   makefile in each task directory sets different flags. For example, the 
   `userprog` kernel compiles code with `#ifdef USERPROG`, and the `devices`
   kernel leaves that code out. The same source files therefore produce
   different `.o` files for each task. If all tasks shared one build directory
   at the repository root, each build would overwrite the other's files. So each
   task directory gets its own `build/`, one per kernel configuration. The
   tests follow the same rule: each kernel's test results go into its own 
   `build/tests/`

   Many larger projects do the same. For example, CMake projects often have
   separate `build-debug/` and `build-releae/` directories for two 
   configurations of the same source. 

   The root `tests/` folder is not a set of tests. It is a small tool that 
   checks whether the whole codebase builds and passes tests from a clean copy.
   It has only two tracked files: `tests/Makefile` and `tests/.gitignore`.
   Both came from the "provided code" commit. 

   ... 
   `make -C tests check`
   1. It removes the old build files in the repo by running `make -c .. distclean`
   2. It makes a sandbox by copying all of `src/` to `tests/<task>/src`. For
      example, it copies it to `tests/<devices>/src`.
   3. It builds the code in that copy
   4. For `devices` and `threads` only, it runs `make chck` int he copy
   5. It checks that `make clean` removes all build files. 
   6. It removes the sandbox. 

   The question names the test by where its SOURCE lives: ... is the 
   C program and checkeer that define the test. It then asks where the LOGS of 
   a RUN go, and that is a different place. The difference is the point of the
   question. A test is written once in `src/tests/`, but each run writes its 
   files into the build directory of the kernel that ran it. The hint, "run this
   test and find out", tells you that you must discover the second location 
   yourself. Your paths are different from the question's path because they answer 
   a different thing. 

   ... one thing to watch in the MCQ, the options may give the same location in 
   a different fom, relative to another directory. For example
   `build/tests/devices/alarm-multiple.output` (from `src/devices`) and 
   `tests/devices/alarm-multiple.output` (from `src/devices/build`) both point
   to the same file as your full path ... Check which dir an 

---

1. `timer_interrupt` (src/devices/timer.c)
   - `timer_init` registers it with `int_register_ext (0x20, ...)`, so it is an
     EXTERNAL interrupt handler. External handlers run with interrupts off. 
   - The timer fires it `TIMER_FREQ` per second (100, in `timer.h`).

2. `thread_tick` 
   - It runs insidde the interrupt handler.
   - It updates the statistics: idle, kernel or user ticks. 
   - It adds 1 to `thread_ticks`, the running thread's ticks in its current 
     slice. When that reaches `TIME_SLICE`
   


   

-- `intr_register_ext` registers an hardware (external) interrupt handler 
   function to a specific interrupt vector number in the CPU's Interrupt
   Descriptor Table (IDT).

   Specifically, in `timer_init`, the call looks like this:

   ```c
   intr_register_ext(0x20, timer_interrupt, "8254 Timer");
   ```

   ... linking the "external" hardware, in this case the timer hardware, onto
   IRQ0, which is where the OS will get the heartbeat from for the rest of its
   runtime. 

-- In PintOS, `struct intr_frame` represents an INTERRUPT FRAME, which is a 
   snapshot of the CPU's register states captured on the thread's stack when an
   interrupt , exception or system call occurs. 

   When the OS transfers execution from a running program to an interrupt 
   handlers, it must preserve the program's exact context so it can resume
   execution later as if nothing happened. 

-- Pre-emption means the act of taking action first to prevent something else
   from happening, or having the first right to buy something before it is 
   offered to others. 

   - GENERAL ACTION: Doing something ahead of time to stop ... 

-- Real reason we enforce preemption and strict time slices is RESPONSIVENESS,
   FAIRNESS, AND SYSTEM SURVIVAL. An OS is a shared environment. If threads 
   could run indefinitely at their own pleasure ... hijack the CPU... the
   OS itself would never get a chance t run again ... Preemption is the OS 
   ensuring no single process can hold the machine hsotage. 

   ... exactly what `thread_tick` snippet is doing. By relying on a hardware
   timer interrupt that fires completely independently of the running code, the
   OS can forcefully rip control away from a thread. When a thread exhausts its
   allocated `TIME_SLICE` (4 ticks), `intr_yield_on_return()`is called to hit
   the pause button and put the thread back in the queue. This rapid, 

-- Thread preemption is needed in PintOS to ensure fair CPU time-sharing, 
   prevent infinite loops or CPU-hogging threads from locking up the system, and
   guarantee responsive handling of real-time or high-priority tasks. 

   WHY PREEMPTION IS NEEDED
   - FAIRNESS
   - RESPONSIVENESS
   - SYSTEM STABILITY... 

   ... THE TICK HANDLER: On every timer tick (`timer_interrupt`), Pintos checks
   if the running thread has used up its time slice. If so, it invokes 
   `thread_yield()` to force a context switch and let another ready thread run. 

---

   
4. `intr_handler` (src/threads/interrupt.c)
   - The assembly stubs in `intr-stubs.S` call this function FOR EVERY INTERRUPT
     . It is the function that called `timer_interrupt` in step 1. 
   - Before the handler: for an external interrupt, it sets 
     `in_external_intr = true` and `yield_on_return = false`
   - It calls the registered handler, here `timer_interrupt`
   - After the handler: it sets `in_external_intr = false`, and calls 
     `pic_end_of_interrupt` so the PIC can send the next interrupt. 
   - Then, if `yield_on_return` is true, it calls `thread_yield ()`. Because
     `in_external_intr` is now false, the `ASSERT (!intr_context ())` in 
     `thread_yield` passes. Interrupts are still off. 

5. `thread_yield` 
   - It turns interrupts off and save 

   ... `intr_context` ... simply just returns `in_external_intr`
   `ASSERT (!intr_context());` ... ensures that our environment is currently not
   in an interrupt mode

---

   -- In Pintos, `thread_yield()` transitions the currently running thread
      from the RUNNING state back to READY state, allowing other threads to 
      have  achance to run. 

      ...

      .. to set a thread's status to `THREAD_DYING0`, you use `thread_exit()`

        

   In PintOS, `frame->vec_no` refers to the INTERRUPT VECTOR NUMBER. 

   It represents the specific ID number assigned to a CPU interrupt or exception
   . The CPU uses this numbeer to index into the INTERRUPT DESCRIPTOR TABLE 
   (IDT) to find the correct handler function to execute. 

   ... 

   COMMON EXAMPLES:
   - 0x20 (32): The System Timer interrupt.
   - 0x21 (33): The Keyboard interrupt
   - 0x26 (38): The Disk/Storage interrupt

   WHY THIS MATTERS
      If `external` evaluates to `true`, it tells the OS that the interrupt was
      triggered by hardware (like a key press or a timer tick). This is crucial
      because Pintos treats external interrupts differently--for instance, 
      THREAD SWITCHING AND SLEEPING are handled strictly to prevent the OS from
      crashing while servicing a hardware device. 

   When a thread's time slice expires and `intr_yield_on_return()` triggers
   `thread_yield()`, the thread is simply paused. The scheduler changes the
   thread's state from `THREAD` ...

4. `intr_handler`
   - The assembly stubs in `intr-stubs.S` call this function for EVERY interrupt. 
     It is the function that called `timer_iinterrupt` in step 1.
   - Before the handler: for an external input, it sets ... 


   When a hardware device triggers an external interrupt, the PIC automatically
   blocks further interrupts to prevent overwhelming the CPU. The CPU then runs
   the appropriate interrupt handler code. 

   At the very end of processing that specific interrupt, `intr_handler()` calls
   `pic_end_of_interrupt()` exactly once. This sends a specific command back to
   the PIC hardware saying, "I have finished handling the current interrupt; 
   you now have permission to send the next one in line..."

   In OS theory, there's a strict mechanical difference between
   "yielding" and "sleeping" (blocking)...

   - SLEEPING `thread-block()`: The thread is taken off the CPU, its state is
     changed to `THREAD_BLOCKED`

   Because a yielding thread never actually leaves the ready queue, when the 
   scheduler looks at the queue a microsecond later to pick the next thread, it
   evaluates all available options. If the yielding thread still happens to be
   the highest priority thread, or if it is the only thread in the system, the
   scheduler will instantly pick it and run it again... 